# 第15章 - 虚拟机
# Chapter 15 - Virtual Machines

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

1. 理解虚拟机 (Virtual Machine) 的概念和用途
2. 区分系统虚拟机 (System VM) 和进程虚拟机 (Process VM)
3. 理解 Hypervisor 的两种类型及其区别
4. 了解 JVM 作为进程虚拟机的工作原理
5. 拓展：Docker 容器 vs 虚拟机、云计算中的虚拟化

---
## 1. 什么是虚拟机 (Virtual Machine)?

### 1.1 核心概念

**虚拟机 (Virtual Machine, VM)** 是通过软件模拟出来的计算机系统。它在一台物理计算机上创造出一个或多个"虚拟的计算机"，每个虚拟机都表现得像一台独立的真实计算机。

**为什么需要虚拟机？** 想象以下场景：

1. **你是一个开发者**：需要同时在 Windows、Linux、macOS 上测试软件，难道买三台电脑？
2. **你是一个公司的 IT 管理员**：有10台服务器，但每台只用了10%的性能，太浪费了！
3. **你是一个安全研究员**：需要运行可疑软件，不能在自己的电脑上直接运行！
4. **你是一个学生**：想学 Linux 但不敢装双系统怕搞坏电脑。

**虚拟机就是这些问题的解决方案！**

### 1.2 虚拟机的好处 (Benefits)

| 好处 | 英文 | 解释 |
|------|------|------|
| 硬件整合 | Hardware consolidation | 一台物理机运行多个VM，提高资源利用率 |
| 隔离性 | Isolation | 每个VM独立运行，一个崩溃不影响其他 |
| 快照/恢复 | Snapshot/Recovery | 可以保存VM状态，随时恢复 |
| 跨平台 | Cross-platform | 在一个OS上运行另一个OS |
| 安全测试 | Security testing | 在隔离环境中安全地运行可疑程序 |
| 成本节约 | Cost saving | 减少物理服务器数量，降低电费和维护成本 |

In [ ]:
# 可视化: 虚拟机的概念 - 一台物理机上运行多个虚拟机

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 左图: 传统方式 - 每个OS一台物理机
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('传统方式: 每个任务一台服务器', fontsize=14, fontweight='bold', color='#D32F2F')

servers = [
    (0.5, '服务器1\nWeb Server', '#42A5F5', '利用率: 15%'),
    (3.5, '服务器2\nDatabase', '#66BB6A', '利用率: 20%'),
    (6.5, '服务器3\nEmail', '#FFA726', '利用率: 10%'),
]

for x, label, color, usage in servers:
    # 物理机
    rect = mpatches.FancyBboxPatch((x, 1), 2.8, 7,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#ECEFF1', edgecolor='#888', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(x + 1.4, 7, label, fontsize=10, ha='center', va='center', fontweight='bold')
    ax1.text(x + 1.4, 5.5, 'Operating\nSystem', fontsize=9, ha='center', va='center')
    # 使用率条
    bar_bg = mpatches.Rectangle((x + 0.4, 2), 2, 1.5, facecolor='#E0E0E0', edgecolor='#888')
    ax1.add_patch(bar_bg)
    pct = float(usage.split(':')[1].strip().replace('%', '')) / 100
    bar_fill = mpatches.Rectangle((x + 0.4, 2), 2 * pct, 1.5, facecolor=color)
    ax1.add_patch(bar_fill)
    ax1.text(x + 1.4, 2.75, usage, fontsize=9, ha='center', va='center', fontweight='bold')
    ax1.text(x + 1.4, 1.2, 'Hardware', fontsize=8, ha='center', color='#666')

ax1.text(5, 0.3, '3台服务器, 总利用率只有 ~15% -- 巨大的浪费!',
         fontsize=10, ha='center', color='#D32F2F', fontweight='bold')

# 右图: 虚拟化方式
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('虚拟化方式: 一台服务器运行多个VM', fontsize=14, fontweight='bold', color='#2E7D32')

# 物理机底层
rect_hw = mpatches.FancyBboxPatch((0.5, 0.5), 9, 1.5,
                                   boxstyle='round,pad=0.1',
                                   facecolor='#ECEFF1', edgecolor='#888', linewidth=2)
ax2.add_patch(rect_hw)
ax2.text(5, 1.25, 'Hardware (一台物理服务器)', fontsize=11, ha='center', va='center', fontweight='bold')

# Hypervisor
rect_hyp = mpatches.FancyBboxPatch((0.5, 2.2), 9, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#7E57C2', edgecolor='white', linewidth=2)
ax2.add_patch(rect_hyp)
ax2.text(5, 2.8, 'Hypervisor (虚拟机管理程序)', fontsize=12, ha='center', va='center',
         fontweight='bold', color='white')

# VMs
vms = [
    (0.8, 'VM 1\nWeb Server\n(Linux)', '#42A5F5'),
    (3.6, 'VM 2\nDatabase\n(Windows)', '#66BB6A'),
    (6.4, 'VM 3\nEmail\n(Linux)', '#FFA726'),
]

for x, label, color in vms:
    rect = mpatches.FancyBboxPatch((x, 3.8), 2.8, 4.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax2.add_patch(rect)
    ax2.text(x + 1.4, 6.5, label, fontsize=11, ha='center', va='center',
            fontweight='bold', color='white')
    ax2.text(x + 1.4, 4.8, 'Guest OS\n+ Apps', fontsize=9, ha='center', va='center', color='white')

# 使用率
rect_usage = mpatches.FancyBboxPatch((1.5, 8.7), 7, 0.8,
                                      boxstyle='round,pad=0.05',
                                      facecolor='#C8E6C9', edgecolor='#2E7D32')
ax2.add_patch(rect_usage)
ax2.text(5, 9.1, '总利用率: ~70% -- 资源充分利用!', fontsize=11, ha='center',
         va='center', fontweight='bold', color='#2E7D32')

plt.tight_layout()
plt.show()

---
## 2. 系统虚拟机 vs 进程虚拟机

虚拟机分为两大类，它们的目标和工作方式完全不同：

### 2.1 系统虚拟机 (System Virtual Machine)

**目标：** 模拟一台完整的计算机，包括硬件、操作系统和一切。

**特点：**
- 可以运行完整的操作系统 (Guest OS)
- 每个VM有自己的虚拟CPU、内存、硬盘、网卡
- 通过 Hypervisor 管理

**例子：** VMware, VirtualBox, Hyper-V, KVM

**为什么存在？** 硬件整合、隔离、跨平台测试

### 2.2 进程虚拟机 (Process Virtual Machine)

**目标：** 为单个程序提供一个平台无关的运行环境。

**特点：**
- 不模拟整个操作系统，只为一个程序创建运行环境
- 程序编译为中间代码 (Intermediate Code)，由虚拟机解释/编译执行
- 实现"一次编写，到处运行" (Write Once, Run Anywhere)

**例子：** JVM (Java Virtual Machine), CLR (.NET), Python 解释器

**为什么存在？** 跨平台兼容性、安全沙箱

In [ ]:
# 可视化: 系统虚拟机 vs 进程虚拟机对比

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# 左图: 系统虚拟机
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 12)
ax1.axis('off')
ax1.set_title('系统虚拟机 (System VM)', fontsize=16, fontweight='bold', color='#1565C0')

layers_sys = [
    (0.5, 0.5, 9, 1.5, '#78909C', 'Hardware\n(物理硬件: CPU, RAM, Disk)'),
    (0.5, 2.2, 9, 1.3, '#7E57C2', 'Hypervisor\n(虚拟机管理程序)'),
]
for x, y, w, h, color, label in layers_sys:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(x + w/2, y + h/2, label, fontsize=10, ha='center', va='center',
            fontweight='bold', color='white')

# VMs
vm_data = [
    (0.7, 'VM 1', '#42A5F5', 'Windows\nGuest OS', 'App A\nApp B'),
    (3.7, 'VM 2', '#66BB6A', 'Linux\nGuest OS', 'App C'),
    (6.7, 'VM 3', '#FFA726', 'macOS\nGuest OS', 'App D\nApp E'),
]
for x, name, color, os_text, app_text in vm_data:
    # VM 框
    rect = mpatches.FancyBboxPatch((x, 3.8), 2.6, 7.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#F5F5F5', edgecolor=color, linewidth=3)
    ax1.add_patch(rect)
    ax1.text(x + 1.3, 10.5, name, fontsize=11, ha='center', fontweight='bold', color=color)
    # Guest OS
    rect_os = mpatches.FancyBboxPatch((x + 0.2, 4.2), 2.2, 2.5,
                                       boxstyle='round,pad=0.05',
                                       facecolor=color, edgecolor='white', alpha=0.7)
    ax1.add_patch(rect_os)
    ax1.text(x + 1.3, 5.45, os_text, fontsize=9, ha='center', va='center',
            fontweight='bold', color='white')
    # Apps
    rect_app = mpatches.FancyBboxPatch((x + 0.2, 7.0), 2.2, 2.5,
                                        boxstyle='round,pad=0.05',
                                        facecolor='#E8EAF6', edgecolor='#888')
    ax1.add_patch(rect_app)
    ax1.text(x + 1.3, 8.25, app_text, fontsize=9, ha='center', va='center')

ax1.text(5, 11.5, '每个VM运行完整OS + 应用程序', fontsize=11, ha='center', color='#1565C0')

# 右图: 进程虚拟机
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 12)
ax2.axis('off')
ax2.set_title('进程虚拟机 (Process VM)', fontsize=16, fontweight='bold', color='#2E7D32')

layers_proc = [
    (0.5, 0.5, 9, 1.5, '#78909C', 'Hardware'),
    (0.5, 2.2, 9, 1.5, '#5C6BC0', 'Host OS (宿主操作系统)'),
]
for x, y, w, h, color, label in layers_proc:
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax2.add_patch(rect)
    ax2.text(x + w/2, y + h/2, label, fontsize=10, ha='center', va='center',
            fontweight='bold', color='white')

# Process VMs (JVM instances)
jvm_data = [
    (0.7, 'JVM\nInstance 1', '#66BB6A', 'Java\nProgram A'),
    (3.7, 'JVM\nInstance 2', '#66BB6A', 'Java\nProgram B'),
    (6.7, 'Python\nInterpreter', '#FFA726', 'Python\nScript'),
]
for x, vm_name, color, prog in jvm_data:
    # VM 层
    rect = mpatches.FancyBboxPatch((x, 4.0), 2.6, 3.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', linewidth=2, alpha=0.7)
    ax2.add_patch(rect)
    ax2.text(x + 1.3, 5.75, vm_name, fontsize=10, ha='center', va='center',
            fontweight='bold', color='white')
    # 程序
    rect_prog = mpatches.FancyBboxPatch((x + 0.2, 7.8), 2.2, 2.5,
                                         boxstyle='round,pad=0.05',
                                         facecolor='#E8EAF6', edgecolor='#888')
    ax2.add_patch(rect_prog)
    ax2.text(x + 1.3, 9.05, prog, fontsize=10, ha='center', va='center')

ax2.text(5, 11.5, '不需要Guest OS, 只为程序提供运行环境', fontsize=11, ha='center', color='#2E7D32')
ax2.text(5, 11.0, '"一次编写, 到处运行" (Write Once, Run Anywhere)', fontsize=10,
         ha='center', color='#555', style='italic')

plt.tight_layout()
plt.show()

---
## 3. Hypervisor (虚拟机管理程序)

Hypervisor 是系统虚拟机的核心组件，负责创建和管理虚拟机。

### 3.1 Type 1 Hypervisor (裸金属型 / Bare-metal)

**直接运行在硬件上**，不需要宿主操作系统。

- 也叫"裸金属" Hypervisor
- 性能更好（没有宿主OS的额外开销）
- 主要用于服务器和数据中心
- 例子：VMware ESXi, Microsoft Hyper-V (Server), Xen, KVM

### 3.2 Type 2 Hypervisor (托管型 / Hosted)

**运行在宿主操作系统之上**，像普通应用程序一样安装。

- 需要先安装一个操作系统，再安装 Hypervisor
- 性能较差（多了一层宿主OS的开销）
- 主要用于个人电脑上的开发/测试
- 例子：VirtualBox, VMware Workstation, Parallels

### 为什么要区分 Type 1 和 Type 2？

**场景决定选择：**
- 数据中心要运行数百个VM → Type 1（性能是关键）
- 开发者在笔记本上测试软件 → Type 2（方便是关键）

In [ ]:
# 可视化: Type 1 vs Type 2 Hypervisor 架构对比

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

def draw_layer(ax, x, y, w, h, color, text, fontsize=10):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.08',
                                    facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, fontsize=fontsize, ha='center', va='center',
            fontweight='bold', color='white')

# Type 1
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 11)
ax1.axis('off')
ax1.set_title('Type 1: Bare-metal Hypervisor', fontsize=15, fontweight='bold', color='#1565C0')

draw_layer(ax1, 0.5, 0.5, 9, 1.2, '#78909C', 'Hardware (物理硬件)')
draw_layer(ax1, 0.5, 2.0, 9, 1.5, '#7E57C2', 'Type 1 Hypervisor\n直接运行在硬件上', 11)

# VMs on Type 1
for i, (x, color, name) in enumerate([(0.8, '#42A5F5', 'VM 1'), (3.5, '#66BB6A', 'VM 2'), (6.2, '#FFA726', 'VM 3')]):
    draw_layer(ax1, x, 3.8, 2.6, 1.5, color, f'{name}\nGuest OS', 9)
    draw_layer(ax1, x, 5.5, 2.6, 2.0, '#90A4AE', f'Apps', 10)

ax1.annotate('没有宿主OS层\n= 更高性能!', xy=(5, 2.5), xytext=(5, 8.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2),
            fontsize=12, ha='center', color='#1565C0', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#BBDEFB', edgecolor='#1565C0'))

ax1.text(5, 10.0, '用途: 服务器, 数据中心, 云计算', fontsize=11, ha='center', color='#555')
ax1.text(5, 9.4, '例: VMware ESXi, Hyper-V Server, Xen', fontsize=10, ha='center', color='#888')

# Type 2
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 11)
ax2.axis('off')
ax2.set_title('Type 2: Hosted Hypervisor', fontsize=15, fontweight='bold', color='#D32F2F')

draw_layer(ax2, 0.5, 0.5, 9, 1.2, '#78909C', 'Hardware (物理硬件)')
draw_layer(ax2, 0.5, 2.0, 9, 1.5, '#5C6BC0', 'Host OS (宿主操作系统)\n如 Windows/macOS/Linux', 10)
draw_layer(ax2, 0.5, 3.8, 9, 1.2, '#7E57C2', 'Type 2 Hypervisor (作为一个应用程序)', 10)

# VMs on Type 2
for i, (x, color, name) in enumerate([(0.8, '#42A5F5', 'VM 1'), (3.5, '#66BB6A', 'VM 2'), (6.2, '#FFA726', 'VM 3')]):
    draw_layer(ax2, x, 5.3, 2.6, 1.2, color, f'{name}\nGuest OS', 9)
    draw_layer(ax2, x, 6.7, 2.6, 1.5, '#90A4AE', f'Apps', 10)

ax2.annotate('多了Host OS层\n= 额外开销', xy=(5, 2.75), xytext=(5, 9.2),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2),
            fontsize=12, ha='center', color='#D32F2F', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#FFCDD2', edgecolor='#D32F2F'))

ax2.text(5, 10.0, '用途: 个人开发, 测试, 学习', fontsize=11, ha='center', color='#555')
ax2.text(5, 9.4, '例: VirtualBox, VMware Workstation', fontsize=10, ha='center', color='#888')

plt.tight_layout()
plt.show()

---
## 4. JVM - Java 虚拟机 (进程虚拟机的经典案例)

### 4.1 JVM 是如何工作的？

Java 的核心理念是 **"Write Once, Run Anywhere"** (一次编写，到处运行)。

**为什么需要 JVM？**

传统编译：
```
C语言源码 → 编译器 → Windows可执行文件(.exe)
                    → 要在Linux上运行？得重新编译！
```

Java 的方式：
```
Java源码(.java) → Java编译器 → 字节码(.class) → JVM解释执行
                                                    ↓
                                          Windows JVM → 运行
                                          Linux JVM   → 运行
                                          macOS JVM   → 运行
```

**关键概念：**
- **字节码 (Bytecode):** Java编译器生成的中间代码，不是任何真实CPU的机器码
- **JVM:** 读取字节码并翻译成当前平台的机器码来执行
- **JIT编译器 (Just-In-Time Compiler):** 运行时将热点字节码编译为本地机器码，提高性能

In [ ]:
# 可视化: JVM 工作流程

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('JVM 工作流程: Write Once, Run Anywhere', fontsize=18, fontweight='bold')

# 源码
draw_box = lambda x, y, w, h, c, t, fs=10: (
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h,
                 boxstyle='round,pad=0.1', facecolor=c, edgecolor='white', linewidth=2)),
    ax.text(x+w/2, y+h/2, t, fontsize=fs, ha='center', va='center', fontweight='bold',
           color='white' if c not in ['#FFF9C4', '#E8EAF6', '#F5F5F5'] else '#333')
)

# Step 1: Java Source
draw_box(0.5, 4, 2.5, 2, '#42A5F5', 'Hello.java\n(Java Source)', 11)

# Arrow
ax.annotate('', xy=(3.5, 5), xytext=(3.0, 5),
           arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Step 2: Java Compiler
draw_box(3.5, 4, 2.5, 2, '#7E57C2', 'javac\n(Java Compiler)', 11)

# Arrow
ax.annotate('', xy=(6.5, 5), xytext=(6.0, 5),
           arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Step 3: Bytecode
draw_box(6.5, 4, 2.5, 2, '#FFA726', 'Hello.class\n(Bytecode\n字节码)', 10)

# Arrows to different JVMs
for y_target, label, color in [(7.5, 'Windows JVM', '#1565C0'), (4, 'Linux JVM', '#2E7D32'), (0.5, 'macOS JVM', '#78909C')]:
    ax.annotate('', xy=(10, y_target + 0.8), xytext=(9.0, 5),
               arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
    draw_box(10, y_target, 2.5, 1.8, color, label, 10)
    
    # Arrow to execution
    ax.annotate('', xy=(13.5, y_target + 0.9), xytext=(12.5, y_target + 0.9),
               arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
    draw_box(13.5, y_target + 0.1, 2, 1.5, '#66BB6A', 'Running!\n(执行)', 10)

# 关键信息
ax.text(8, 9.2, '同一份字节码 (.class) 可在任何平台的JVM上运行!',
        fontsize=13, ha='center', color='#D32F2F', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FFCDD2', edgecolor='#D32F2F', alpha=0.9))

# 标注阶段
ax.text(1.75, 3.3, 'Step 1\n编写', fontsize=9, ha='center', color='#555')
ax.text(4.75, 3.3, 'Step 2\n编译', fontsize=9, ha='center', color='#555')
ax.text(7.75, 3.3, 'Step 3\n分发', fontsize=9, ha='center', color='#555')
ax.text(11.25, 9.6, 'Step 4: JVM解释/JIT编译执行', fontsize=9, ha='center', color='#555')

plt.tight_layout()
plt.show()

In [ ]:
# Python本身也是一种进程虚拟机!
# 让我们看看Python的字节码

import dis

def add_numbers(a, b):
    result = a + b
    return result

print("Python 源码:")
print("  def add_numbers(a, b):")
print("      result = a + b")
print("      return result")
print()
print("Python 字节码 (Bytecode):")
print("=" * 50)
dis.dis(add_numbers)
print("=" * 50)
print()
print("解释:")
print("  LOAD_FAST   - 将变量加载到栈上")
print("  BINARY_ADD  - 从栈上取两个值相加")
print("  STORE_FAST  - 将结果存入变量")
print("  RETURN_VALUE - 返回栈顶值")
print()
print("这些字节码不是真实CPU能执行的机器码!")
print("它们由 Python 解释器 (CPython VM) 逐条解释执行。")
print("所以 Python 解释器也是一种进程虚拟机!")

---
## 5. 拓展：Docker 容器 vs 虚拟机

### 为什么要了解这个？
Docker 是现代软件开发中最重要的技术之一，面试经常被问到。

### 核心区别

| 特性 | 虚拟机 (VM) | Docker 容器 (Container) |
|------|------------|------------------------|
| 虚拟化级别 | 硬件级 | 操作系统级 |
| 是否需要Guest OS | 是 (每个VM一个完整OS) | 否 (共享宿主内核) |
| 启动时间 | 分钟级 | 秒级 |
| 资源占用 | 大 (GB级内存) | 小 (MB级内存) |
| 隔离性 | 强 (完全隔离) | 较弱 (共享内核) |
| 适用场景 | 需要不同OS, 强隔离 | 快速部署, 微服务 |

**类比：**
- VM = 每个租户一栋独立别墅（有自己的地基、水电系统）
- Container = 同一栋公寓楼的不同房间（共享地基和水电，但各自独立）

In [ ]:
# 可视化: VM vs Docker 容器

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# VM方式
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 12)
ax1.axis('off')
ax1.set_title('Virtual Machine', fontsize=16, fontweight='bold', color='#1565C0')

def add_box(ax, x, y, w, h, color, text, fs=10):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.05',
                                    facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, text, fontsize=fs, ha='center', va='center',
            fontweight='bold', color='white' if color not in ['#FFF9C4','#F5F5F5','#E8EAF6'] else '#333')

add_box(ax1, 0.5, 0.5, 9, 1, '#78909C', 'Infrastructure (Hardware)')
add_box(ax1, 0.5, 1.7, 9, 1, '#5C6BC0', 'Host Operating System')
add_box(ax1, 0.5, 2.9, 9, 1, '#7E57C2', 'Hypervisor')

for i, (x, c) in enumerate([(0.7, '#42A5F5'), (3.5, '#66BB6A'), (6.3, '#FFA726')]):
    add_box(ax1, x, 4.2, 2.6, 1.5, c, f'Guest OS\n(~2GB)', 9)
    add_box(ax1, x, 5.9, 2.6, 1, '#E8EAF6', 'Libs/Bins', 9)
    add_box(ax1, x, 7.1, 2.6, 1.2, '#90A4AE', f'App {i+1}', 10)

ax1.text(5, 9.0, '每个VM都有完整Guest OS', fontsize=12, ha='center', color='#D32F2F', fontweight='bold')
ax1.text(5, 8.4, '启动时间: 分钟级 | 内存: GB级', fontsize=10, ha='center', color='#888')

# Docker方式
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 12)
ax2.axis('off')
ax2.set_title('Docker Container', fontsize=16, fontweight='bold', color='#2E7D32')

add_box(ax2, 0.5, 0.5, 9, 1, '#78909C', 'Infrastructure (Hardware)')
add_box(ax2, 0.5, 1.7, 9, 1, '#5C6BC0', 'Host Operating System')
add_box(ax2, 0.5, 2.9, 9, 1, '#00897B', 'Docker Engine')

# 没有Guest OS!
for i, (x, c) in enumerate([(0.7, '#42A5F5'), (3.5, '#66BB6A'), (6.3, '#FFA726')]):
    add_box(ax2, x, 4.2, 2.6, 1, '#E8EAF6', 'Libs/Bins', 9)
    add_box(ax2, x, 5.4, 2.6, 1.5, '#90A4AE', f'App {i+1}', 10)

ax2.annotate('没有Guest OS层!\n共享宿主内核', xy=(5, 4.0), xytext=(5, 8.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2),
            fontsize=13, ha='center', color='#2E7D32', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#C8E6C9', edgecolor='#2E7D32'))

ax2.text(5, 7.8, '启动时间: 秒级 | 内存: MB级', fontsize=10, ha='center', color='#888')

plt.tight_layout()
plt.show()

### 5.1 云计算中的虚拟机

**为什么云计算离不开虚拟化？**

当你使用 AWS、Azure、阿里云时，你"租"的服务器其实就是虚拟机：

1. **弹性扩展** - 需要更多算力？几分钟内创建新VM
2. **按需付费** - 只为实际使用的资源付费
3. **资源隔离** - 不同用户的VM互不影响
4. **迁移方便** - VM可以从一台物理机迁移到另一台

**一个有趣的事实：** 你现在使用的几乎所有互联网服务（微信、淘宝、B站），背后都运行在虚拟机或容器上！

In [ ]:
# 模拟: 简单的虚拟机资源分配器

class SimpleHypervisor:
    """一个简化的 Hypervisor 模拟器，演示资源分配"""
    
    def __init__(self, total_cpu_cores, total_ram_gb, total_disk_gb):
        self.total_cpu = total_cpu_cores
        self.total_ram = total_ram_gb
        self.total_disk = total_disk_gb
        self.used_cpu = 0
        self.used_ram = 0
        self.used_disk = 0
        self.vms = []
    
    def create_vm(self, name, cpu, ram, disk):
        """创建一个虚拟机"""
        if (self.used_cpu + cpu > self.total_cpu or
            self.used_ram + ram > self.total_ram or
            self.used_disk + disk > self.total_disk):
            print(f"  [FAILED] 资源不足, 无法创建 {name}!")
            return False
        
        self.used_cpu += cpu
        self.used_ram += ram
        self.used_disk += disk
        self.vms.append({'name': name, 'cpu': cpu, 'ram': ram, 'disk': disk, 'status': 'running'})
        print(f"  [OK] 创建 {name}: {cpu} cores, {ram}GB RAM, {disk}GB Disk")
        return True
    
    def show_status(self):
        """显示资源使用情况"""
        print(f"\n{'='*60}")
        print(f"Hypervisor 资源状态")
        print(f"{'='*60}")
        print(f"CPU:  {self.used_cpu}/{self.total_cpu} cores ({self.used_cpu/self.total_cpu*100:.0f}% 使用)")
        print(f"RAM:  {self.used_ram}/{self.total_ram} GB ({self.used_ram/self.total_ram*100:.0f}% 使用)")
        print(f"Disk: {self.used_disk}/{self.total_disk} GB ({self.used_disk/self.total_disk*100:.0f}% 使用)")
        print(f"\n运行中的虚拟机 ({len(self.vms)} 台):")
        for vm in self.vms:
            print(f"  - {vm['name']}: {vm['cpu']} cores, {vm['ram']}GB RAM, {vm['disk']}GB Disk [{vm['status']}]")

# 模拟场景: 一台物理服务器
print("物理服务器: 32 cores, 128GB RAM, 2000GB Disk")
print("正在创建虚拟机...\n")

hypervisor = SimpleHypervisor(32, 128, 2000)

hypervisor.create_vm("Web-Server-01", 4, 16, 100)
hypervisor.create_vm("Database-01", 8, 32, 500)
hypervisor.create_vm("App-Server-01", 4, 16, 200)
hypervisor.create_vm("App-Server-02", 4, 16, 200)
hypervisor.create_vm("Dev-Test", 2, 8, 100)
hypervisor.create_vm("Monitoring", 2, 4, 50)

# 尝试创建一个超大的VM
print("\n尝试创建一个大VM:")
hypervisor.create_vm("Big-Analytics", 16, 64, 1000)

hypervisor.show_status()

---
## 6. 练习题 (Exercises)

### 练习 1: 概念辨析 (4分)

判断以下描述对应的是系统虚拟机 (System VM) 还是进程虚拟机 (Process VM)：

1. VirtualBox 中运行 Ubuntu
2. Java 程序在 JVM 上运行
3. Python 脚本在 CPython 解释器中执行
4. 云服务器上的一个实例
5. Android 应用在 ART (Android Runtime) 中运行
6. VMware ESXi 上的 Windows Server

### 练习 2: Hypervisor 对比 (6分)

1. 解释 Type 1 和 Type 2 Hypervisor 的主要区别。(3分)
2. 一家银行需要在数据中心虚拟化其服务器。推荐 Type 1 还是 Type 2？给出两个理由。(3分)

In [ ]:
# 练习 3: 资源计算
# 一台物理服务器有 64 个CPU核心, 256GB RAM, 4TB 磁盘
# 需要运行以下虚拟机:

vm_requirements = [
    {"name": "Web Server x3", "count": 3, "cpu": 4, "ram": 8, "disk": 50},
    {"name": "Database", "count": 1, "cpu": 16, "ram": 64, "disk": 1000},
    {"name": "App Server x4", "count": 4, "cpu": 4, "ram": 16, "disk": 100},
    {"name": "Dev/Test x2", "count": 2, "cpu": 2, "ram": 8, "disk": 200},
]

# TODO: 计算总共需要多少CPU, RAM, Disk
# TODO: 判断这台物理服务器是否够用
# TODO: 如果不够, 还需要多少资源?

total_cpu = 0
total_ram = 0
total_disk = 0

print("虚拟机资源需求分析")
print("=" * 60)
for vm in vm_requirements:
    cpu = vm['count'] * vm['cpu']
    ram = vm['count'] * vm['ram']
    disk = vm['count'] * vm['disk']
    total_cpu += cpu
    total_ram += ram
    total_disk += disk
    print(f"{vm['name']}: {cpu} cores, {ram}GB RAM, {disk}GB Disk")

print(f"\n总需求: {total_cpu} cores, {total_ram}GB RAM, {total_disk}GB Disk")
print(f"物理机: 64 cores, 256GB RAM, 4000GB Disk")
print(f"\n请分析: 这台物理服务器是否够用? 如果不够还需要什么?")

### 练习 4: 思考题

1. **解释为什么 JVM 被称为进程虚拟机而不是系统虚拟机。** (3分)

2. **一个公司正在考虑将其应用从虚拟机迁移到Docker容器。讨论这样做的两个好处和两个风险。** (4分)

3. **为什么说"云计算的本质就是虚拟化"？请从技术角度解释。** (3分)

---

### 本节小结 (Summary)

| 概念 | 关键点 |
|------|--------|
| 虚拟机 | 通过软件模拟的计算机系统, 提供隔离和资源整合 |
| 系统VM | 模拟完整计算机, 可运行完整OS, 由Hypervisor管理 |
| 进程VM | 为单个程序提供运行环境, 如JVM, 实现跨平台 |
| Type 1 Hypervisor | 裸金属型, 直接运行在硬件上, 性能更好 |
| Type 2 Hypervisor | 托管型, 运行在宿主OS上, 使用更方便 |
| JVM | Java进程虚拟机, 字节码 + JIT编译, 跨平台 |
| Docker容器 | OS级虚拟化, 共享内核, 更轻量更快 |